# Mangalate · servidor de tradução no Google Colab (GPU grátis)

Este notebook liga o servidor que o app **Mangalate** usa para detectar os balões, apagar o texto original, reconstruir a arte e diagramar a tradução.

1. Menu **Ambiente de execução › Alterar tipo** → escolha **GPU T4**.
2. Preencha as chaves na célula 1 (ou cadastre em *Secrets* do Colab com os mesmos nomes).
3. Rode as células na ordem. No final aparece o **endereço** e a **chave** para colar no app em *Configurações › Mangalate*.

A sessão gratuita do Colab desliga depois de algumas horas ou quando fica ociosa; é só rodar de novo (o endereço muda).

In [ ]:
#@title 1 · Chaves
MANGALATE_KEY = ""  #@param {type:"string"}
DEEPSEEK_API_KEY = ""  #@param {type:"string"}
GEMINI_API_KEY = ""  #@param {type:"string"}
GROQ_API_KEY = ""  #@param {type:"string"}
OPENAI_API_KEY = ""  #@param {type:"string"}

import os, secrets
try:
    from google.colab import userdata
except Exception:
    userdata = None

def pick(name, value):
    if value:
        return value
    if userdata is not None:
        try:
            return userdata.get(name) or ""
        except Exception:
            return ""
    return ""

env = {
    "DEEPSEEK_API_KEY": pick("DEEPSEEK_API_KEY", DEEPSEEK_API_KEY),
    "GEMINI_API_KEY": pick("GEMINI_API_KEY", GEMINI_API_KEY),
    "GROQ_API_KEY": pick("GROQ_API_KEY", GROQ_API_KEY),
    "OPENAI_API_KEY": pick("OPENAI_API_KEY", OPENAI_API_KEY),
}
os.environ.update({k: v for k, v in env.items() if v})
os.environ["MANGALATE_KEY"] = pick("MANGALATE_KEY", MANGALATE_KEY) or secrets.token_urlsafe(18)
print("Motores com chave:", [k for k, v in env.items() if v] or "nenhum (use Offline/NLLB no app)")

In [ ]:
#@title 2 · Instalar (5–10 min na primeira vez)
!nvidia-smi -L || echo "Sem GPU: troque o tipo de ambiente para T4"
%cd /content
!test -d manga-image-translator || git clone --depth 1 https://github.com/zyddnys/manga-image-translator.git
%cd /content/manga-image-translator
!pip install -q -r requirements.txt
!pip install -q httpx uvicorn fastapi
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
!wget -q -O /content/mangalate_gateway.py https://raw.githubusercontent.com/delkrusty/Mangalate/main/server/colab/mangalate_gateway.py
print("Pronto")

In [ ]:
#@title 3 · Ligar o servidor e gerar o endereço
import subprocess, time, re, socket, os

def wait_port(port, timeout):
    t0 = time.time()
    while time.time() - t0 < timeout:
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(3)
    return False

os.chdir("/content/manga-image-translator")
log = open("/content/translator.log", "w")
translator = subprocess.Popen(
    ["python", "server/main.py", "--start-instance", "--host=127.0.0.1", "--port=8001", "--use-gpu"],
    stdout=log, stderr=subprocess.STDOUT,
)
print("Carregando modelos...")
if not wait_port(8001, 900):
    raise SystemExit("O servidor não subiu. Veja /content/translator.log")

gateway = subprocess.Popen(["python", "/content/mangalate_gateway.py"], env=os.environ.copy())
wait_port(5003, 60)

tlog = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:5003"],
    stdout=tlog, stderr=subprocess.STDOUT,
)
url = None
for _ in range(60):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break

print("\n================ MANGALATE ================")
print("Endereço do servidor:", url or "(veja /content/tunnel.log)")
print("Chave de acesso     :", os.environ["MANGALATE_KEY"])
print("===========================================")
print("Cole os dois no app: Configurações › Mangalate › Servidor de tradução")

In [ ]:
#@title 4 · Manter ligado (deixe rodando enquanto lê)
import time
try:
    while True:
        time.sleep(60)
        if translator.poll() is not None:
            print("O tradutor parou. Últimas linhas do log:")
            print(open("/content/translator.log").read()[-3000:])
            break
except KeyboardInterrupt:
    pass